# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Tue Nov 10, 2026 - Day 21: Machine learning - decision trees and forests (on Zoom)

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix
from ipywidgets import interactive_output, IntSlider, FloatSlider, SelectionSlider, HBox, VBox, Layout
from IPython.display import display
plt.rcParams.update({'font.size': 13, 'figure.figsize': (9.4, 3.6)})   # 9.4 in fits 1280 px; 7.4 in beside widget controls

import threading                                    # lock matplotlib's math-text parser (slider redraws run on another thread)
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

def controls_right(fn, **widgets):
    for w in widgets.values():
        w.layout = Layout(width='230px')
        w.style = {'description_width': '95px'}
    display(HBox([interactive_output(fn, widgets), VBox(list(widgets.values()), layout=Layout(justify_content='center', width='240px'))]))

# ---- the data: ELAsTiCC (DESC's simulated Rubin alert stream), type Ia and type II supernovae only.
# One row per supernova: 84 numbers ("features") computed from its simulated r- and i-band light curves
# (fits of the Bazin and Villar light-curve shapes, plus summary statistics of the magnitudes), and its true type.
# Same file as ASTR 496 (Fall 2025) week 11. Simulations: portal.nersc.gov/cfs/lsst/DESC_TD_PUBLIC/ELASTICC/
df = pd.read_csv('data/day21_elasticc_ia_ii.csv.bz2')
y_all = (df.pop('type') == 'Ia').values             # True = type Ia, False = type II
X_all = df
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.3, random_state=0)

# ---- two features we can draw: how fast it fades in i (mag per day), and the r-band fall time (days, log scale)
F1, F2 = 'mag_linear_fit_slope_lssti', 'fullflux_bazin_fit_fall_time_lsstr'
LAB1, LAB2 = 'i-band fading rate (mag/day)', 'r-band fall time, $\\log_{10}$ days'
def two(X):
    return np.c_[X[F1].values, np.log10(X[F2].clip(lower=0.1).values)]
Z_tr, Z_te = two(X_tr), two(X_te)
N_SMALL = 1000                                      # train the drawable trees on 1000 supernovae, so overfitting shows
Zs, ys = Z_tr[:N_SMALL], y_tr[:N_SMALL]

DEPTHS = list(range(1, 16))
trees = {d: DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=0).fit(Zs, ys) for d in DEPTHS}   # entropy, as on the slides (sklearn's default is Gini)
acc_tr = [trees[d].score(Zs, ys) for d in DEPTHS]
acc_te = [trees[d].score(Z_te, y_te) for d in DEPTHS]
cv_acc = [cross_val_score(DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=0), Zs, ys, cv=5).mean() for d in DEPTHS]   # training set only
CV_BEST = DEPTHS[int(np.argmax(cv_acc))]                # the depth cross-validation picks

NTREES = [1, 2, 5, 10, 30, 100, 300]
forests = {n: RandomForestClassifier(n_estimators=n, criterion='entropy', random_state=0, n_jobs=-1).fit(Zs, ys) for n in NTREES}
acc_forest = [forests[n].score(Z_te, y_te) for n in NTREES]

# ---- all 84 features, all 7496 training supernovae (about 10 s)
tree_all = DecisionTreeClassifier(criterion='entropy', random_state=0).fit(X_tr, y_tr)
forest_all = RandomForestClassifier(n_estimators=300, criterion='entropy', random_state=0, n_jobs=-1).fit(X_tr, y_tr)
boost_all = GradientBoostingClassifier(n_estimators=300, random_state=0).fit(X_tr, y_tr)
boost_curve = [np.mean(p == y_te) for p in boost_all.staged_predict(X_te)]
cm = confusion_matrix(y_te, forest_all.predict(X_te), labels=[True, False])   # rows: true Ia, true II

# ---- k nearest neighbors on the same two features, with and without putting both on the same scale
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score
KS = [1, 3, 5, 9, 15, 31, 51, 101, 151]
scaler = StandardScaler().fit(Zs)                    # subtract each feature's mean, divide by its standard deviation
knn = {(k, sc): KNeighborsClassifier(k).fit(scaler.transform(Zs) if sc else Zs, ys) for k in KS for sc in (False, True)}
def knn_acc(k, sc, Z, y):
    return knn[(k, sc)].score(scaler.transform(Z) if sc else Z, y)
knn_tr = {sc: [knn_acc(k, sc, Zs, ys) for k in KS] for sc in (False, True)}
knn_te = {sc: [knn_acc(k, sc, Z_te, y_te) for k in KS] for sc in (False, True)}

# ---- the 300-tree forest on two features gives a probability, not just a label
p_te = forests[300].predict_proba(Z_te)[:, 1]        # the fraction of the 300 trees that vote Ia
fpr, tpr, _ = roc_curve(y_te, p_te)
AUC = roc_auc_score(y_te, p_te)

# ---- what if the labels come only from the brightest supernovae (the ones that get spectra)?
MAG_R = 'mag_weighted_mean_lsstr'                     # mean r magnitude
m_tr, m_te = X_tr[MAG_R].values, X_te[MAG_R].values
faint_te = m_te > np.median(m_te)                    # the faint half of the test set
FRACS = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
_rng = np.random.default_rng(0)
biased = {}
for fr in FRACS:
    n = int(fr * len(m_tr))
    for kind, idx in [('bright', np.argsort(m_tr)[:n]), ('random', _rng.choice(len(m_tr), n, replace=False))]:
        f = RandomForestClassifier(n_estimators=100, criterion='entropy', random_state=0, n_jobs=-1).fit(X_tr.iloc[idx], y_tr[idx])
        biased[(fr, kind)] = dict(acc=f.score(X_te[faint_te], y_te[faint_te]), ia=y_tr[idx].mean(), cut=m_tr[idx].max())

print(f"{len(df)} supernovae ({y_all.sum()} Ia, {(~y_all).sum()} II), {X_all.shape[1]} features; "
      f"test accuracy: one tree {tree_all.score(X_te, y_te):.3f}, forest {forest_all.score(X_te, y_te):.3f}, "
      f"boosting {boost_curve[-1]:.3f}")


In [ ]:
# Plotting functions for the slides (a "skip" cell)
BLUE, ORANGE = '#1f77b4', '#ff7f0e'                 # Ia blue, II orange, on every plot today
CMAP = ListedColormap(['#ffd8b0', '#c6dbef'])

def plot_recap():
    """Last Tuesday's simulated year of kilonovae, with the symbols fixed: M is the peak absolute magnitude."""
    rng = np.random.default_rng(0)
    R, Mbar, r_out = 56.0, -14.816, 1500.0
    n = rng.poisson(R * 1e-9 * 0.5 * 4 / 3 * np.pi * r_out**3)
    r = r_out * rng.uniform(size=n)**(1 / 3)
    M = rng.normal(Mbar, 0.5, n)
    found = M + 5 * np.log10(r * 1e6) - 5 <= 24.0
    fig, ax = plt.subplots(figsize=(7.4, 3.3))
    ax.scatter(r[~found], M[~found], s=6, c='0.75', label=f'missed ({(~found).sum()})')
    ax.scatter(r[found], M[found], s=14, c='C0', label=f'found ({found.sum()})')
    rr = np.linspace(50, 1500, 200)
    ax.plot(rr, 24.0 - 5 * np.log10(rr * 1e6) + 5, 'k--', lw=1, label='$m = 24.0$, i.e. $M = 24.0 - \\mu(d)$')
    ax.axhline(Mbar, color='C3', ls=':', label='$\\bar{M} = -14.8$, the population mean')
    ax.set(xlabel='distance (Mpc)', ylabel='peak absolute mag $M$', ylim=(-13, -17.2))
    ax.legend(fontsize=9, loc='upper left', ncol=2)
    plt.tight_layout(); plt.show()

def plot_bazin():
    t = np.linspace(-30, 150, 400)
    def bazin(t, rise, fall):
        return np.exp(-t / fall) / (1 + np.exp(-t / rise))
    fig, ax = plt.subplots(figsize=(7.4, 3.3))
    for rise, fall, c, lab in [(3, 22, BLUE, 'Ia-like: falls in ~22 days'), (5, 58, ORANGE, 'II-like: falls in ~58 days')]:
        f = bazin(t, rise, fall)
        ax.plot(t, f / f.max(), c=c, lw=2, label=lab)
    ax.set(xlabel='days', ylabel='flux (scaled to peak)', ylim=(0, 1.1))
    ax.legend(fontsize=10)
    plt.tight_layout(); plt.show()

def scatter_two(ax, Z, y, n=1500, s=5):
    ax.scatter(Z[:n][~y[:n], 0], Z[:n][~y[:n], 1], s=s, c=ORANGE, alpha=0.6, label='type II')
    ax.scatter(Z[:n][y[:n], 0], Z[:n][y[:n], 1], s=s, c=BLUE, alpha=0.6, label='type Ia')
    ax.set(xlabel=LAB1, ylabel=LAB2, xlim=(-0.015, 0.05), ylim=(0.3, 3.2))

def plot_two_features():
    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.6))
    scatter_two(ax[0], Z_tr, y_tr)
    ax[0].legend(fontsize=10, loc='upper right', markerscale=3)
    bins = np.linspace(-0.015, 0.05, 50)
    ax[1].hist(Z_tr[~y_tr, 0], bins=bins, color=ORANGE, alpha=0.5)
    ax[1].hist(Z_tr[y_tr, 0], bins=bins, color=BLUE, alpha=0.5)
    ax[1].set(xlabel=LAB1, ylabel='supernovae', title='the fading rate alone (all 7,496 training)')
    ax[1].title.set_fontsize(11)
    plt.tight_layout(); plt.show()

def plot_first_tree():
    fig, ax = plt.subplots(figsize=(9.4, 3.9))
    boxes = plot_tree(trees[2], feature_names=['fading rate', 'log fall time'], class_names=['II', 'Ia'], filled=True,
                      impurity=False, proportion=False, precision=3, fontsize=10, ax=ax)
    for b in boxes:                                 # 'value = [II, Ia]' counts, written out as whole numbers
        b.set_text(re.sub(r'value = \[([\d.]+), ([\d.]+)\]', lambda m: f'II: {float(m[1]):.0f}, Ia: {float(m[2]):.0f}', b.get_text()).replace('samples', 'supernovae'))
    plt.tight_layout(); plt.show()

def entropy(p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

def gain_curve(x, y, cuts):
    H0, out = entropy(y.mean()), []
    for c in cuts:
        L = x <= c
        if L.all() or (~L).all():
            out.append(0.0); continue
        out.append(H0 - L.mean() * entropy(y[L].mean()) - (~L).mean() * entropy(y[~L].mean()))
    return np.array(out)

CUTS = np.round(np.arange(-0.010, 0.0401, 0.001), 3)
GAIN = gain_curve(Zs[:, 0], ys, CUTS)

def plot_split(cut=0.000, save=None):
    x = Zs[:, 0]
    L = x <= cut
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.4))
    bins = np.linspace(-0.015, 0.05, 45)
    ax[0].hist(x[~ys], bins=bins, color=ORANGE, alpha=0.6, label='II')
    ax[0].hist(x[ys], bins=bins, color=BLUE, alpha=0.6, label='Ia')
    ax[0].axvline(cut, color='k', lw=2)
    ax[0].set(xlabel=LAB1, ylabel='supernovae',
              title=f'left: {ys[L].mean() if L.any() else 0:.0%} Ia | right: {ys[~L].mean() if (~L).any() else 0:.0%} Ia', )
    ax[0].title.set_fontsize(11)
    ax[0].legend(fontsize=9)
    g = gain_curve(x, ys, [cut])[0]
    ax[1].plot(CUTS, GAIN, 'k')
    ax[1].plot(cut, g, 'o', c='C3', ms=9)
    ax[1].set(xlabel='where we cut', ylabel='information gain (bits)', ylim=(0, 0.6), title=f'gain = {g:.2f} bits')
    ax[1].title.set_fontsize(11)
    plt.tight_layout()
    if save:
        return fig
    plt.show()

def split_controls():
    controls_right(plot_split, cut=FloatSlider(value=0.000, min=-0.010, max=0.040, step=0.001, readout_format='.3f',
                                                description='cut (mag/day)'))

GX, GY = np.meshgrid(np.linspace(-0.015, 0.05, 300), np.linspace(0.3, 3.2, 300))
GRID = np.c_[GX.ravel(), GY.ravel()]

def regions(ax, model):
    ax.pcolormesh(GX, GY, model.predict(GRID).reshape(GX.shape).astype(int), cmap=CMAP, shading='auto', rasterized=True)

def plot_depth(depth=2, save=None):
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.4))
    regions(ax[0], trees[depth])
    scatter_two(ax[0], Zs, ys, n=N_SMALL, s=4)
    ax[0].set_title(f'depth {depth}: {trees[depth].get_n_leaves()} leaves', fontsize=11)
    ax[1].plot(DEPTHS, acc_tr, 'k-', label='training set')
    ax[1].plot(DEPTHS, acc_te, 'C3-', label='test set (a peek, to show overfitting)')
    ax[1].axvline(depth, color='0.6', ls=':')
    ax[1].set(xlabel='maximum depth', ylabel='accuracy', ylim=(0.8, 1.005))
    ax[1].legend(fontsize=9, loc='lower right')
    ax[1].set_title(f'train {acc_tr[depth - 1]:.3f}, test {acc_te[depth - 1]:.3f}', fontsize=11)
    plt.tight_layout()
    if save:
        return fig
    plt.show()

def depth_controls():
    controls_right(plot_depth, depth=IntSlider(value=2, min=1, max=15, step=1, description='max depth'))

def plot_cv():
    best = CV_BEST
    fig, ax = plt.subplots(figsize=(7.4, 3.2))
    ax.plot(DEPTHS, acc_tr, 'k-', label='training set')
    ax.plot(DEPTHS, cv_acc, 'C2-o', ms=4, label='5-fold cross-validation (training set only)')
    ax.plot(best, acc_te[best - 1], '*', c='C3', ms=15, mec='k', label=f'test set, used once, at depth {best}: {acc_te[best - 1]:.3f}')
    ax.axvline(best, color='C2', ls=':')
    ax.set(xlabel='maximum depth', ylabel='accuracy', ylim=(0.8, 1.005), title=f'cross-validation picks depth {best}')
    ax.legend(fontsize=9, loc='lower right')
    plt.tight_layout(); plt.show()

def plot_two_bootstraps():
    rng = np.random.default_rng(3)
    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.5), sharey=True)
    for k in range(2):
        i = rng.integers(0, N_SMALL, N_SMALL)                 # resample the 1000 with replacement (Day 3's bootstrap)
        t = DecisionTreeClassifier(criterion='entropy', random_state=0).fit(Zs[i], ys[i])
        regions(ax[k], t)
        scatter_two(ax[k], Zs, ys, n=N_SMALL, s=3)
        ax[k].set_title(f'bootstrap sample {k + 1}: test accuracy {t.score(Z_te, y_te):.3f}', fontsize=11)
    ax[1].set_ylabel('')
    plt.tight_layout(); plt.show()

def plot_forest(n_trees=1, save=None):
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.4))
    regions(ax[0], forests[n_trees])
    scatter_two(ax[0], Zs, ys, n=N_SMALL, s=4)
    ax[0].set_title(f'{n_trees} tree, grown all the way' if n_trees == 1 else f'{n_trees} trees, each grown all the way', fontsize=11)
    ax[1].semilogx(NTREES, acc_forest, 'C0-o', ms=4, label='forest')
    ax[1].axhline(acc_te[CV_BEST - 1], color='C3', ls='--', label=f'depth-{CV_BEST} tree (chosen by cross-validation)')
    ax[1].axvline(n_trees, color='0.6', ls=':')
    ax[1].set(xlabel='number of trees', ylabel='test accuracy', ylim=(0.88, 0.96))
    ax[1].legend(fontsize=9, loc='lower right')
    ax[1].set_title(f'test accuracy {forests[n_trees].score(Z_te, y_te):.3f}', fontsize=11)
    plt.tight_layout()
    if save:
        return fig
    plt.show()

def forest_controls():
    controls_right(plot_forest, n_trees=SelectionSlider(options=NTREES, value=1, description='trees'))

def plot_ladder():
    rng = np.random.default_rng(1)
    fig, ax = plt.subplots(1, 3, figsize=(9.4, 3.0))
    x = np.sort(rng.uniform(0, 10, 25)); yl = 1.2 * x + 2 + rng.normal(0, 1.2, 25)
    ax[0].errorbar(x, yl, 1.2, fmt='ko', ms=3, lw=1)
    ax[0].plot([0, 10], [2, 14], 'C0', lw=2)
    ax[0].set_title('we write $y = ax + b$', fontsize=11)
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel
    yg = np.sin(x) + 0.5 * np.sin(2.3 * x) + rng.normal(0, 0.15, 25)
    gp = GaussianProcessRegressor(RBF(1.0) + WhiteKernel(0.02), random_state=0).fit(x[:, None], yg)
    xx = np.linspace(0, 10, 200); mu, sd = gp.predict(xx[:, None], return_std=True)
    ax[1].plot(x, yg, 'ko', ms=3); ax[1].plot(xx, mu, 'C0', lw=2); ax[1].fill_between(xx, mu - 2 * sd, mu + 2 * sd, color='C0', alpha=0.2)
    ax[1].set_title('a GP: we pick the kernel', fontsize=11)
    regions(ax[2], trees[3]); scatter_two(ax[2], Zs, ys, n=400, s=3)
    ax[2].set(xlabel='', ylabel=''); ax[2].set_title('a tree: cuts fit to the data', fontsize=11)
    for a in ax:
        a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()

def plot_vote():
    rng = np.random.default_rng(5)
    fig, ax = plt.subplots(1, 4, figsize=(9.4, 2.7), sharey=True)
    for k in range(3):
        i = rng.integers(0, N_SMALL, N_SMALL)
        regions(ax[k], DecisionTreeClassifier(criterion='entropy', max_features=1, random_state=k).fit(Zs[i], ys[i]))
        ax[k].set_title(f'tree {k + 1}', fontsize=11)
    from matplotlib.colors import LinearSegmentedColormap
    pv = forests[300].predict_proba(GRID)[:, 1].reshape(GX.shape)     # fraction of the 300 trees voting Ia
    ax[3].pcolormesh(GX, GY, pv, cmap=LinearSegmentedColormap.from_list('vote', ['#ffd8b0', 'white', '#c6dbef']),
                     vmin=0, vmax=1, shading='auto', rasterized=True)
    ax[3].set_title('vote of 300 trees', fontsize=11)
    for a in ax:
        scatter_two(a, Zs, ys, n=N_SMALL, s=2)
        a.set_xticks([]); a.set_yticks([]); a.set_xlabel('fading rate', fontsize=10); a.set_ylabel('')
    ax[0].set_ylabel('log fall time', fontsize=10)
    plt.tight_layout(); plt.show()

def plot_knn(k=15, scaled=True, save=None):
    m = knn[(k, scaled)]
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.4))
    G = scaler.transform(GRID) if scaled else GRID
    ax[0].pcolormesh(GX, GY, m.predict(G).reshape(GX.shape).astype(int), cmap=CMAP, shading='auto', rasterized=True)
    scatter_two(ax[0], Zs, ys, n=N_SMALL, s=4)
    ax[0].set_title(f'k = {k}, ' + ('features rescaled' if scaled else 'raw features'), fontsize=11)
    ax[1].semilogx(KS, knn_tr[scaled], 'k-', label='training set')
    ax[1].semilogx(KS, knn_te[scaled], 'C3-', label='test set')
    ax[1].axvline(k, color='0.6', ls=':')
    ax[1].set(xlabel='k (neighbors that vote)', ylabel='accuracy', ylim=(0.85, 1.005))
    ax[1].legend(fontsize=9, loc='upper right')
    ax[1].set_title(f'test {knn_acc(k, scaled, Z_te, y_te):.3f}', fontsize=11)
    plt.tight_layout()
    if save:
        return fig
    plt.show()

def knn_controls():
    from ipywidgets import Checkbox
    controls_right(plot_knn, k=SelectionSlider(options=KS, value=51, description='k'),
                   scaled=Checkbox(value=False, description='rescale'))

def plot_threshold(t=0.5, save=None):
    called = p_te >= t
    purity, compl = y_te[called].mean(), called[y_te].mean()
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.4))
    bins = np.linspace(0, 1, 26)
    ax[0].hist(p_te[~y_te], bins=bins, color=ORANGE, alpha=0.6, label='true II')
    ax[0].hist(p_te[y_te], bins=bins, color=BLUE, alpha=0.6, label='true Ia')
    ax[0].axvline(t, color='k', lw=2)
    ax[0].set(xlabel='fraction of trees voting Ia', ylabel='supernovae', yscale='log')
    ax[0].set_title(f'call Ia above {t:.2f}: purity {purity:.2f}, completeness {compl:.2f}', fontsize=10)
    ax[0].legend(fontsize=9, loc='upper center')
    ax[1].plot(fpr, tpr, 'C0')
    ax[1].plot((~y_te & called).sum() / (~y_te).sum(), compl, 'o', c='C3', ms=9)
    ax[1].plot([0, 1], [0, 1], 'k:', lw=1)
    ax[1].set(xlabel='fraction of II called Ia', ylabel='fraction of Ia called Ia', title=f'ROC curve (area {AUC:.3f})')
    ax[1].title.set_fontsize(11)
    plt.tight_layout()
    if save:
        return fig
    plt.show()

def threshold_controls():
    controls_right(plot_threshold, t=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, readout_format='.2f', description='threshold'))

def plot_biased(frac=0.2, save=None):
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.4))
    bins = np.linspace(18, 26, 50)
    ax[0].hist(m_tr[~y_tr], bins=bins, color=ORANGE, alpha=0.6, label='II')
    ax[0].hist(m_tr[y_tr], bins=bins, color=BLUE, alpha=0.6, label='Ia')
    cut = biased[(frac, 'bright')]['cut']
    ax[0].axvspan(bins[0], cut, color='0.85', zorder=0, label='gets a spectrum')
    ax[0].set(xlabel='mean r magnitude (fainter to the right)', ylabel='supernovae')
    ax[0].set_title(f'labeled: brightest {frac:.0%}, {biased[(frac, "bright")]["ia"]:.0%} of them Ia', fontsize=10)
    ax[0].legend(fontsize=9, loc='upper right'); ax[0].set_ylim(0, 640)
    ax[1].plot(FRACS, [biased[(fr, 'random')]['acc'] for fr in FRACS], 'k-o', ms=4, label='random training set')
    ax[1].plot(FRACS, [biased[(fr, 'bright')]['acc'] for fr in FRACS], 'C3-o', ms=4, label='brightest only')
    ax[1].axvline(frac, color='0.6', ls=':')
    ax[1].set(xlabel='fraction of the training set labeled', ylabel='accuracy on the faint half', ylim=(0.88, 1.0))
    ax[1].legend(fontsize=9, loc='lower right')
    ax[1].set_title(f"faint half: {biased[(frac, 'bright')]['acc']:.3f} vs {biased[(frac, 'random')]['acc']:.3f}", fontsize=10)
    plt.tight_layout()
    if save:
        return fig
    plt.show()

def biased_controls():
    controls_right(plot_biased, frac=SelectionSlider(options=FRACS[:-1], value=0.2, description='labeled'))

def plot_boost():
    fig, ax = plt.subplots(figsize=(7.4, 3.2))
    ax.plot(np.arange(1, len(boost_curve) + 1), boost_curve, 'C2', label='boosting: trees of depth 3, added one at a time')
    ax.axhline(forest_all.score(X_te, y_te), color='C0', ls='--', label='random forest, 300 trees')
    ax.axhline(tree_all.score(X_te, y_te), color='C3', ls=':', label='one full-depth tree')
    ax.set(xlabel='number of trees', ylabel='test accuracy', ylim=(0.95, 1.0), title='all 84 features, all 7496 training supernovae')
    ax.legend(fontsize=9, loc='lower right')
    plt.tight_layout(); plt.show()

NICE = {'mag_linear_fit_slope_lssti': 'i fading rate (line fit)', 'mag_linear_fit_slope_lsstr': 'r fading rate (line fit)', 'fullflux_bazin_fit_fall_time_lssti': 'i fall time (Bazin)',
        'fullflux_bazin_fit_fall_time_lsstr': 'r fall time (Bazin)', 'mag_bins_window1.0_offset0.0_linear_trend_lssti': 'i fading rate (binned)',
        'mag_inter_percentile_range_1_lsstr': 'r magnitude range', 'mag_inter_percentile_range_1_lssti': 'i magnitude range',
        'mag_bins_window1.0_offset0.0_linear_trend_sigma_lssti': 'i fading-rate error (binned)',
        'mag_bins_window1.0_offset0.0_linear_trend_sigma_lsstr': 'r fading-rate error (binned)',
        'mag_chi2_lssti': 'i scatter about the mean', 'fullflux_villar_fit_plateau_duration_lssti': 'i plateau length (fit)'}

def plot_importance():
    order = np.argsort(forest_all.feature_importances_)[-10:]
    fig, ax = plt.subplots(figsize=(7.4, 3.6))
    ax.barh([NICE.get(X_all.columns[i], X_all.columns[i]) for i in order], forest_all.feature_importances_[order], color='C0')
    ax.set(xlabel='importance (share of the entropy drop, all 300 trees)', title='the forest\'s top 10 of 84 features')
    ax.tick_params(axis='y', labelsize=10)
    plt.tight_layout(); plt.show()

def plot_confusion():
    fig, ax = plt.subplots(figsize=(5.0, 3.6))
    ax.imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]}', ha='center', va='center', fontsize=16, color='w' if cm[i, j] > cm.max() / 2 else 'k')
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Ia', 'II'], yticklabels=['Ia', 'II'], xlabel='the forest says', ylabel='true type')
    plt.tight_layout(); plt.show()

def plot_rare():
    tpr = cm[0, 0] / cm[0].sum()                         # fraction of true Ia called Ia
    fpr = cm[1, 0] / cm[1].sum()                         # fraction of true II called Ia
    f = np.logspace(-3, np.log10(0.5), 200)              # fraction of the sample that is Ia
    fig, ax = plt.subplots(figsize=(7.4, 3.3))
    ax.semilogx(f, f * tpr + (1 - f) * (1 - fpr), 'C0', label='forest: accuracy')
    ax.semilogx(f, 1 - f, 'k--', label='"always say II": accuracy')
    ax.semilogx(f, f * tpr / (f * tpr + (1 - f) * fpr), 'C3', label='forest: purity (fraction of its "Ia" that are Ia)')
    ax.axvline(0.01, color='0.6', ls=':', lw=1)
    ax.set(xlabel='fraction of the sample that is type Ia', ylabel='fraction', ylim=(0, 1.02))
    ax.legend(fontsize=9, loc='lower right')
    plt.tight_layout(); plt.show()


## What did ABC learn from last Tuesday's kilonovae?

On Zoom: unmute or use the chat any time.

<img src="images/day20_static_summary_widget.png" alt="Last Tuesday's summary widget at three settings: with the count only, the kept draws lie along a diagonal ridge; adding the median magnitude keeps the same ridge; adding the median distance leaves a small blob around the true value" style="display:block;margin:0 auto;max-height:250px;max-width:100%">

* we drew $\theta$ from the prior, simulated, and kept $\theta$ when the summaries matched within $\epsilon$
* the count alone left a ridge, and the median distance broke it

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Today, the model learns which numbers matter from examples whose answer we already know.**

</div>

## $\mu$ is the distance modulus, and $\bar{M}$ is the mean peak magnitude

* in this course $\mu = m - M = 5\log_{10}(d / 10\,{\rm pc})$ is always the distance modulus (last week's plots used it for $\bar{M}$)
* $M$ is one kilonova's peak absolute magnitude, and $\bar{M} \approx -14.8$ is the population mean

In [ ]:
# last Tuesday's simulated year, relabeled: M is each kilonova's peak, M-bar the population mean
plot_recap()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**From now on, $\mu$ only ever means the distance modulus!**

</div>

## Today - what if we can't write the model down at all?

* machine learning: give it examples with known answers, and let it build the model
* nearest neighbors, then decision trees and forests of them, on simulated Rubin supernovae (type Ia or type II?)
* how to test a model, and which mistakes it makes
* where astronomers use trees, and where their training labels come from

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Can a model learned from examples tell a type Ia supernova from a type II?**

</div>

## So far we wrote down the model - with ML, it's built from labeled examples

* Days 2-19: we wrote the model (a line, an orbit, a GP kernel, a graph) and fit its parameters
* last week we had no likelihood, but we still wrote the simulator

In [ ]:
# the same idea three ways: a line we wrote, a GP, and a decision tree on today's supernovae
plot_ladder()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We didn't write down an equation for the tree on the right!**

</div>

## Supervised learning has labels, unsupervised learning finds groups without them

<img src="images/day21_unsup_vs_sup_pair.png" alt="Two panels. Left, unsupervised: points in x and y with no labels, grouped into three ellipses by a clustering model. Right, supervised classification: blue and orange labeled points, split first along x and then along y by a tree" style="display:block;margin:0 auto;max-height:290px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">ASTR 496 (Fall 2025) course slides.</div>

* on the left there are no labels, and the model finds groups (Thursday)
* on the right every point has a label, and the model learns to predict it (today)
* a label that's a category is **classification**, a label that's a number is **regression**

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The "supervision" is the training set of labeled examples that we provide.**

</div>

## Why classify supernovae? Rubin will find far more than we can take spectra of

<img src="images/rubin_observatory.jpg" alt="Vera C. Rubin Observatory on Cerro Pachon at sunset, the dome open" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">NSF-DOE Vera C. Rubin Observatory. Credit: O. Bonin/SLAC National Accelerator Laboratory (CC BY 4.0).</div>

* Rubin will send millions of alerts a night (an alert is a notice that something in the sky changed)
* a spectrum tells us what each one is, but only a tiny fraction will ever get one

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**For almost everything Rubin finds, the light curve is all we'll have to classify it.**

</div>

## Type Ia supernovae fade steadily, while many type II stay bright for weeks

<img src="images/day21_real_lcs_2011fe_2020jfo.png" alt="Real light curves in r and i, magnitudes below peak against days since peak. Left, the type Ia SN 2011fe rises in about two weeks and fades steadily, with a Bazin curve fit to each band. Right, the type II-P SN 2020jfo stays within half a magnitude of peak for about 40 days, then drops" style="display:block;margin:0 auto;max-height:270px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Left: SN 2011fe, CfA r′ and i′ photometry. Right: SN 2020jfo, Pan-STARRS r and i (Young Supernova Experiment). Both are real observations (ELAsTiCC is simulated).</div>

* a type Ia is an exploding white dwarf whose peak brightness, once corrected for how fast it fades, is nearly the same every time (we use them to measure distances)
* a type II is a massive star whose core collapses, and its hydrogen envelope keeps many on a plateau for weeks (type II-P, right)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A type II in our Ia sample is fainter than a Ia, and biases the distances (you'll see how in Lab 09).**

</div>

## In ELAsTiCC's simulated Rubin alerts, we know every supernova's true type

* ELAsTiCC (the Extended LSST Astronomical Time-series Classification Challenge, where LSST is Rubin's Legacy Survey of Space and Time) is a simulated Rubin alert stream, built to test classifiers
* it was made by DESC (the Dark Energy Science Collaboration)
* we kept 10,709 supernovae: 5,175 type Ia and 5,534 type II
* each one comes with 84 numbers measured from its $r$ and $i$ light curves

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Roughly half Ia and half II is our choice (real samples aren't split evenly like this).**

</div>

## One number computed from a light curve is called a feature

* e.g. fit the Bazin function, $f(t) = A\,e^{-(t - t_0)/\tau_{\rm fall}} / \left(1 + e^{-(t - t_0)/\tau_{\rm rise}}\right)$, to each band
* $A$ sets the height and $t_0$ the time of peak, $\tau_{\rm rise}$ and $\tau_{\rm fall}$ how fast it rises and falls - we keep $\tau_{\rm fall}$

In [ ]:
# the Bazin light-curve shape, at the median fall times of our Ia and II in r
plot_bazin()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Each supernova becomes a row of numbers, and the classifier sees only those numbers.**

</div>

## The data are a matrix $X$ of features, and a column $y$ of labels

<img src="images/day21_data_representation.png" alt="A table: N rows, one per sample, and P columns, one per feature, forming the feature matrix X; beside it a single column of N labels, y" style="display:block;margin:0 auto;max-height:270px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">A. Müller &amp; S. Raschka, scikit-learn tutorial, SciPy 2016 (CC0).</div>

* $N$ rows (supernovae) by $P$ columns (features) - this is the design matrix from Day 9
* scikit-learn wants exactly this: `model.fit(X, y)`, then `model.predict(X_new)`

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We'll give every method today the same $X$ and $y$, and change only the model.**

</div>

## Two of the 84 features already separate most of the Ia from the II

In [ ]:
# left: two of the 84 features for 1500 supernovae; right: just the fading rate
plot_two_features()

* a positive rate means fading (magnitudes grow as a supernova fades)
* Ia (blue) fade faster in $i$, with a narrow range of fall times, and II (orange) spread wide and overlap them

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Look at your data first! (a classifier can only find structure that's there)**

</div>

## How do we know if a model works? Test it on supernovae it never saw

<img src="images/day21_train_test_split_matrix.png" alt="The feature matrix X and labels y each cut into two pieces by rows: most of the rows for training, the rest held back for testing" style="display:block;margin:0 auto;max-height:250px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">A. Müller &amp; S. Raschka, scikit-learn tutorial, SciPy 2016 (CC0).</div>

* we train on 70% (7,496 supernovae, the **training set**) and hold back 30% (3,213) as a **test set**
* **accuracy** is the fraction classified correctly (on the training set it's optimistic - a flexible model can memorize it)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The test set is our proxy for tomorrow night's alerts.**

</div>

## The simplest classifier is a vote of the $k$ nearest neighbors (kNN)

* to classify a new supernova, find the $k$ training supernovae closest to it, and take a majority vote
* there's no fit at all - we just keep the training set
* "closest" is a distance in **feature space**, i.e. the plane of fading rate and log fall time

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**So how do we measure how far apart two supernovae are?**

</div>

## kNN needs the features on the same scale

In [ ]:
# k and rescaling: the boxes (left), training and test accuracy (right)
knn_controls()

<img src="images/day21_static_knn_widget.png" alt="The kNN widget at k = 51. Left, raw features: the boundary is a horizontal band, ignoring the fading rate; test accuracy 0.901. Right, rescaled features: the boundary uses both features; test accuracy 0.933" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* it opens unscaled: log fall time spans ~3 and the fading rate only ~0.06 mag/day, and the distance barely sees the fading rate (now tick rescale)
* rescaling (subtract each feature's mean, divide by its standard deviation) takes $k = 51$ from 0.901 to 0.933

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If your method measures a distance, rescale the features first!**

</div>

## Can we classify with a series of yes/no questions?

* each **node** of a decision tree asks one question of one feature, e.g. "fading rate ≤ 0.016?" (yes goes left)
* the answer sends the supernova down a **branch**, and the last box (a **leaf**) gives the class
* the **depth** is the most questions in a row

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A tree's leaves are boxes in feature space (each cut is on one feature).**

</div>

## A depth-2 tree asks each supernova at most two questions

In [ ]:
# a depth-2 tree on the two features
plot_first_tree()

* to make the mistakes easy to see, the two-feature models (kNN too) train on just 1,000 of the 7,496
* each box shows its question, how many supernovae reach it, and how many are [II, Ia]

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**You can read the classification straight off the tree!**

</div>

## OK, so how do we pick where to cut?

<img src="images/day21_entropy_twoclass.png" alt="Entropy of a two-class set against the fraction of plus signs, p(+): zero at p = 0 and p = 1, where the set is all one class, and a maximum of 1 bit at p = 0.5, where it is evenly mixed" style="display:block;margin:0 auto;max-height:250px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">ASTR 496 (Fall 2025) course slides.</div>

* the **entropy** of a box with class fractions $p_i$ is $H = -\sum_i p_i \log_2 p_i$ (in bits)
* all one class gives $H = 0$ (pure), and half and half gives $H = 1$ (a coin flip)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We want each cut to leave boxes with less entropy than the box we started with.**

</div>

## The tree keeps the cut with the largest information gain

* a cut splits a box (the parent, with $N$ supernovae) into a left and a right box ($N_L$ and $N_R$ of them)
* information gain $= H_{\rm parent} - \left(\frac{N_L}{N} H_L + \frac{N_R}{N} H_R\right)$, i.e. the parent's entropy minus the weighted mean of the two sides'
* the tree tries every feature and every cut

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We're doing a top-down search (the best cut first, then the same again in each new box).**

</div>

## Our first cut takes the entropy from 1.00 bit to 0.54

* parent: 1,000 supernovae, 48% Ia, so $H = 1.00$ bit
* fading rate ≤ 0.016: 573 supernovae, 16% Ia, $H = 0.63$
* fading rate > 0.016: 427 supernovae, 92% Ia, $H = 0.42$
* the weighted mean of the two sides is $0.573 \times 0.63 + 0.427 \times 0.42 = 0.54$, and the gain is $1.00 - 0.54 = 0.46$ bits

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**One cut takes us from a coin flip to two boxes that are 84% II and 92% Ia.**

</div>

## Cuts near either end of the fading rate gain almost nothing

In [ ]:
# the cut on the fading rate: class mix on each side (left), and the gain (right)
split_controls()

<img src="images/day21_static_split_widget.png" alt="The split widget at two cuts. Left, a cut at 0.000 mag per day: nearly everything is on the right and the gain is small. Right, a cut at 0.016: the left side is mostly type II, the right mostly type Ia, at the peak of the gain curve" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* slide the cut from 0 to 0.04, and watch the gain (right) rise and fall

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The gain peaks at 0.016 mag/day, right where our tree made its first cut.**

</div>

## A deeper tree fits the training set better, and the test set worse

In [ ]:
# max depth: the tree's boxes (left), and training and test accuracy (right)
depth_controls()

<img src="images/day21_static_depth_widget.png" alt="The depth widget at depth 3 and depth 15. At depth 3 the boxes are few and large. At depth 15 there are 75 small boxes, some around single training points; training accuracy is 0.995 but test accuracy has dropped to 0.912" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* slide the depth from 2 to 15 - the boxes shrink around single points, training accuracy (black) climbs to 0.995, and test accuracy (red) peaks at depth 4, then falls to 0.912

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Fitting the noise in the training set like this is called <em>overfitting</em>.**

</div>

## Remember the bias-variance tradeoff? Depth sets it for a tree

<img src="images/day21_overfitting_cartoon.png" alt="Accuracy against model complexity: training accuracy (blue) keeps rising, generalization accuracy (green) rises to a sweet spot and then falls; underfitting on the left, overfitting on the right" style="display:block;margin:0 auto;max-height:250px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">A. Müller &amp; S. Raschka, scikit-learn tutorial, SciPy 2016 (CC0). "Generalization" = accuracy on new data.</div>

* a shallow tree is biased (too few boxes for the real boundary), and a deep one has high variance (Day 5 - and Day 8's polynomial, with enough terms to hit every point)
* depth is a **hyperparameter**, i.e. we set it before the fit

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**How do we choose the depth without peeking at the test set?**

</div>

## Cross-validation tunes the depth on the training set alone

<img src="images/day21_grid_search_cross_validation.png" alt="The data split into a training set and a test set; the training set is split again into five folds, each fold taking a turn as the validation set while the other four train; the test set is used once at the end" style="display:block;margin:0 auto;max-height:230px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">A. Müller &amp; S. Raschka, scikit-learn tutorial, SciPy 2016 (CC0).</div>

* split the training set into 5 folds, train on 4, score on the fifth (the **validation** fold), and rotate
* if we chose the depth by test accuracy, the test set wouldn't be unseen any more

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Tune with cross-validation, and look at the test set only once, at the end!**

</div>

## Cross-validation picks depth 4, and the test set then gives 0.935

In [ ]:
# training and cross-validation accuracy against depth; the test set once, at the end
plot_cv()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**0.935 is an honest estimate, because the test set played no part in choosing the depth.**

</div>

## What if we resample the data and grow another tree?

In [ ]:
# two full-depth trees, each trained on a bootstrap resample of the same 1,000 supernovae
plot_two_bootstraps()

* resample the same 1,000 supernovae with replacement (Day 3's bootstrap), and grow a full tree on each
* the two trees draw different boxes where Ia and II overlap

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Variance - different data lead to different trees!**

</div>

## A random forest lets many different trees vote

* an **ensemble** is many models combined into one answer
* **bagging** (bootstrap aggregation): grow many full trees, each on its own bootstrap sample, and let them vote
* a **random forest** also gives each cut a random subset of the features (i.e. the trees are less alike)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**There are many possible trees, so we do the thing we always do and average over them (it lowers the variance).**

</div>

## The vote of 300 trees is smoother than any one of them

In [ ]:
# three trees from a forest, and the vote of all 300
plot_vote()

* dots are the 1,000 training supernovae, and the shading is blue where the trees call Ia, orange where they call II (last panel: paler means a closer vote)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A lone training point still wins its own patch, but only a pale one (about 2 trees in 3).**

</div>

## Can a forest beat the best single tree?

In [ ]:
# number of trees: the forest's boxes (left), and test accuracy against the depth-4 tree (right)
forest_controls()

<img src="images/day21_static_forest_widget.png" alt="The forest widget with 1 tree and with 300 trees. One tree gives jagged boxes around single points and 0.904 test accuracy. 300 trees give 0.937, level with the depth-4 tree chosen by cross-validation" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* slide from 1 tree to 300: the jagged boxes smooth out, and test accuracy goes from 0.904 (like the overfit depth-15 tree, 0.912) to 0.937

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**On two features, a forest of untuned trees does as well as the best tree we could tune (0.935).**

</div>

## Boosting grows each new tree to fix the mistakes of the ones before

* a forest grows its trees independently, while boosting adds shallow trees one after another
* each new tree concentrates on what the ensemble so far gets wrong
* the most popular version is called **gradient boosting** (EXTRA at the end)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Boosting adds up many shallow trees into one strong classifier!**

</div>

## With all 84 features, a forest or boosting gets 99% right

* now all 84 features, and all 7,496 training supernovae
* test accuracy: forest 99.2%, boosting 99.3%, one tree 98.2%

In [ ]:
# all 84 features: test accuracy as boosting adds trees, against a forest and one tree
plot_boost()

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**More features help all three methods, and the ensembles still beat one tree.**

</div>

## The most important features are how fast and how long each supernova fades

In [ ]:
# feature importance: how much each feature lowered the entropy, summed over all the forest's cuts
plot_importance()

* our two hand-picked features rank 2nd and 3rd
* fading-rate *errors* rank high too (probably because they track how bright and well measured each one is)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Check that the important features make physical sense (the forest can't!)**

</div>

## Where does the forest go wrong?

In [ ]:
# the test set: rows are the true types, columns the forest's answer
plot_confusion()

* this is a **confusion matrix** for the 84-feature forest on the test set: the diagonal is right, the rest is wrong (11 + 16 of 3,213)
* a Ia called II is lost to cosmology
* a II called Ia is an impostor in the Hubble diagram (distance modulus against redshift - Weyant et al., last Tuesday)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A II called Ia is the worse mistake for cosmology, because it biases the distances.**

</div>

## Purity and completeness count the two kinds of mistake

* the fraction of our Ia calls that really are Ia is the **purity** (also called precision)
* the fraction of true Ia that we call Ia is the **completeness** (also called recall)
* the forest's answer is a probability (the fraction of its 300 trees voting Ia) - we choose the threshold for "Ia"

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Which one does cosmology need more?**

</div>

## A higher threshold buys purity, and costs completeness

In [ ]:
# back to two features. The threshold for calling it Ia: votes per true type (left), ROC curve (right)
threshold_controls()

<img src="images/day21_static_threshold_widget.png" alt="The threshold widget at 0.5 and 0.9. Left: histograms of the fraction of trees voting Ia for true II (orange, piled near 0) and true Ia (blue, piled near 1), with the threshold as a black line; purity 0.92 and completeness 0.95 at 0.5, purity 0.96 and completeness 0.73 at 0.9. Right: the ROC curve, area 0.978, with a red dot at the current threshold" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* back on two features: slide the threshold from 0.5 to 0.9, and purity goes from 0.92 to 0.96 while completeness drops from 0.95 to 0.73
* the **ROC curve** (receiver operating characteristic, right) traces every threshold (its area, the **AUC**, is 1 for a perfect classifier and 0.5 for a coin flip)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Pick the threshold for your science (for cosmology, purity)!**

</div>

## When the class you want is rare, accuracy hides the mistakes

In [ ]:
# the same forest, on samples where the class we want (Ia here) gets rarer and rarer
plot_rare()

* back to the 84-feature forest: our sample is half Ia by our choice - now make the class we want 1 in 100 (kilonovae are far rarer still)
* "always say II" is then 99% accurate, and finds nothing
* the forest is 99% accurate too, but its purity drops to about half

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Don't quote accuracy alone for a rare class! (more on this Thursday)**

</div>

## Where do astronomers use trees? Variable stars, in 25 classes

<img src="images/day21_richards2011_fig5.png" alt="Richards et al. 2011 Fig. 5: a 25 by 25 confusion matrix for variable-star classes, true class across the top and predicted class down the side, from Mira and semiregular variables through Cepheids, RR Lyrae and Delta Scuti stars to eclipsing binaries; most of the weight is on the diagonal, with mistakes among similar pulsators" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Richards et al. 2011, ApJ 733, 10 (arXiv:1101.1959), Fig. 5.</div>

* a random forest on light-curve features (periods, amplitudes and shapes, from one band with no colors)
* a confusion matrix like ours, each column one true class: it was wrong about 1 time in 5 (21-23%)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Many of its mistakes are between classes that are physically similar.**

</div>

## ALeRCE, one of Rubin's alert brokers, sorts alerts with forests

<img src="images/day21_sanchezsaez2021_fig2.png" alt="The ALeRCE taxonomy as a tree: an alert is real or bogus; real ones are transient, stochastic or periodic; transients split into SN Ia, SN Ib/c, SN II and superluminous SN; stochastic into AGN, QSO, blazar, young stellar object and cataclysmic variable; periodic into eclipsing binaries, long-period variables, Cepheids, RR Lyrae, Delta Scuti and others" style="display:block;margin:0 auto;max-height:290px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Sánchez-Sáez et al. 2021, AJ 161, 141 (arXiv:2008.03311), Fig. 2 (illustrations in the figure: @wandering_astro).</div>

* an **alert broker** classifies a survey's alerts as they arrive - ALeRCE does this for the Zwicky Transient Facility, and is one of Rubin's brokers
* its classifier is a tree of random forests: real or "bogus" (an image artifact), then transient, stochastic (randomly varying) or periodic, then 15 classes

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Real alerts go through forests like the ones we just grew.**

</div>

## Trees can predict numbers too, e.g. photometric redshifts

<img src="images/day21_carrascokind2013_pair.png" alt="Two panels from Carrasco Kind and Brunner 2013. Left, Fig. 1: one prediction tree drawn radially, hundreds of colored nodes branching out to leaves. Right, Fig. 4 left: density contours of the photometric redshift from regression trees against the spectroscopic redshift for SDSS galaxies, from 0 to 0.25, clustered along the dashed one-to-one line" style="display:block;margin:0 auto;max-height:280px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Carrasco Kind &amp; Brunner 2013, MNRAS 432, 1483 (arXiv:1303.7269), Fig. 1 (one tree) and Fig. 4, left (SDSS test galaxies).</div>

* a **regression tree** puts a number in each leaf: the mean of the training points that end up there
* TPZ, a forest of them by Matias Carrasco Kind and Robert Brunner (both at Illinois when they built it), predicts a galaxy's redshift from its magnitudes and colors
* right: TPZ's redshift against the redshift from a spectrum

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A tree can't predict past the edge of its training set (each leaf is a training mean).**

</div>

## Where do the labels come from? Spectra, of the brightest objects

<img src="images/day21_dsfp_training_set.png" alt="A schematic: the number of SDSS galaxies against r magnitude; the photometric sample (gray) peaks near 19.5, and the much smaller spectroscopic sample (red) peaks near 18, brighter" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Schematic from A. Miller, LSSTC Data Science Fellowship Program, Session 14 (MIT license).</div>

* so the training set is brighter (and nearer) than the data we want to classify
* this is Day 19's selection effect again, now in the training set

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**So is our training set a fair sample of the supernovae we want to classify?**

</div>

## A forest trained on the brightest 20% does worse on the faint half

In [ ]:
# label only the brightest: who is labeled (left), accuracy on the faint half (right)
biased_controls()

<img src="images/day21_static_biased_widget.png" alt="The biased-training widget at 20% and 50% labeled. Left: r-magnitude histograms of type Ia (blue, brighter) and II (orange, fainter), with the labeled brightest fraction shaded gray; at 20% the labeled set is 80% Ia. Right: accuracy on the faint half of the test set against the fraction labeled, for a random training set (black, about 0.985) and the brightest only (red, 0.933 at 20%, catching up by 50%)" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* slide the labeled fraction from 20% to 50% - at 20% (1,499 supernovae) the labeled set is 80% Ia, and the forest gets 0.933 on the faint half (0.985 from a random 1,499)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The forest makes four to five times as many mistakes on the faint half, with the same number of labels!**

</div>

## All five classifiers get worse with a training set like real follow-up

<img src="images/day21_lochner2016_fig6.png" alt="Lochner et al. 2016 Fig. 6: area under the ROC curve for five algorithms (naive Bayes, k nearest neighbors, a neural network, a support vector machine, boosted decision trees) and four ways of making features; solid lines use a representative training set and sit higher, dashed lines a non-representative one and sit lower; boosted decision trees are highest" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Lochner, McEwen, Peiris, Lahav &amp; Winter 2016, ApJS 225, 31 (arXiv:1603.00882), Fig. 6.</div>

* the AUC for five classifiers: BDT = boosted trees, KNN = our nearest neighbors, and three we haven't met
* solid lines use a training set drawn like the data, dashed lines one like real follow-up, which in their words "perform[s] significantly worse"

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We can't fix an unrepresentative training set by switching classifiers.**

</div>

## PLAsTiCC gave 8,000 objects to learn from, and 3.5 million to classify

<img src="images/day21_hlozek2023_fig1.png" alt="Two treemaps from Hlozek et al. 2023 Fig. 1: the number of objects per class in the PLAsTiCC training sample (left, about 8000) and test sample (right, about 3.5 million); SNe Ia and SNe II dominate both, and rare classes such as kilonovae are tiny boxes" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Hložek, Malz, Ponder et al. 2023, ApJS 267, 25 (arXiv:2012.12392), Fig. 1 (CC BY 4.0).</div>

* PLAsTiCC (2018), ELAsTiCC's predecessor, was a public challenge on Kaggle (a data-science competition site) to classify simulated Rubin light curves
* its training set looked like what spectroscopy can follow up, and the test set was mostly fainter and farther

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**To score well, you had to deal with the biased training set.**

</div>

## The winner used boosted trees, after augmenting the training set

<img src="images/day21_boone2019_fig1.png" alt="Boone 2019 Fig. 1: normalized histograms of true redshift on a log axis for the spectroscopically confirmed (training) set, blue, peaking near 0.2, and the full (test) set, orange, peaking near 0.6" style="display:block;margin:0 auto;max-height:270px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Boone 2019, AJ 158, 257 (arXiv:1907.04690), Fig. 1.</div>

* PLAsTiCC's training set (blue) has median redshift 0.18, the test set (orange) 0.43
* Kyle Boone's Avocado fit each training light curve with a GP (Day 17), redrew it at higher redshift, then classified with boosted trees

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Note how Avocado changed the training set, and kept a standard classifier!**

</div>

## After neural networks, we'll train classifiers on simulations

<img src="images/day20_lueckmann2021_algorithms.png" alt="Four flowcharts from Lueckmann et al. 2021: Monte Carlo ABC next to three methods that train an estimator on the simulations - likelihood estimation, posterior estimation and ratio estimation" style="display:block;margin:0 auto;max-height:210px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Lueckmann, Boelts, Greenberg, Gonçalves &amp; Macke 2021, AISTATS (arXiv:2101.04653), Fig. 1. Last Tuesday's slide.</div>

* Nov 17 and 19 are neural networks, and simulation-based inference (SBI) comes after them (last week I said Thursday)
* one kind of SBI, ratio estimation (right), trains a classifier to tell real ($\theta$, data) pairs from shuffled ones

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**You'll see classifiers like today's again in SBI.**

</div>

## What did we learn today?

* with ML you give it labeled examples and it builds the model (the simplest: kNN, on rescaled features)
* a decision tree picks each cut to maximize the information gain, and we choose its depth by cross-validation
* forests average many bootstrapped trees, and boosting adds trees that fix earlier mistakes (99% of Ia and II right)
* purity and completeness trade off through the threshold, and accuracy hides mistakes in rare classes
* labels come from spectra of bright objects, and an unrepresentative training set hurts every method

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Hold some data back, and judge every model on it!**

</div>

## Colloquium today, 3:45 pm, 134 Astronomy (also online): Jessie Muir (Cincinnati)

<img src="images/speaker_jessie_muir.jpg" alt="Jessie Muir, University of Cincinnati" style="display:block;margin:0 auto;max-height:250px;max-width:100%">

* **"Cosmology with Galaxy Imaging Surveys"**
* what imaging surveys of galaxies (the Dark Energy Survey, then Rubin) tell us about dark energy
* refreshments at 3:30 pm in Astronomy 222 (go!)

<div style="font-size:0.8em; text-align:center; color:#666;">Photo: University of Cincinnati research directory. Listing: astro.illinois.edu/news-events/astrophysics-colloquium; calendars.illinois.edu event 33565869.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Rubin will image both supernovae like today's and the galaxies in her talk.**

</div>

## Before you go

* Lab 09 (ABC on a contaminated supernova Hubble diagram) is posted today, due **Tue Nov 17 by Noon** (fork then PR, as always - every date is in `ASSIGNMENTS.md`)
* **Quiz 5 is Thu Nov 12**, at the start of class: ABC (Nov 3) and today's machine learning
* no lab defenses today - if you were due to defend, I'll email you to set up a short one-on-one instead
* Thursday: imbalanced classes, outliers, and unsupervised learning
* EXTRA: the Gini impurity, and how boosting weights its trees

## EXTRA: the Gini impurity usually picks nearly the same cuts as entropy

* the Gini impurity of a box is $G = 1 - \sum_i p_i^2$: 0 when pure, 0.5 for half and half (two classes)
* it's the chance that two objects drawn at random from the box have different classes
* scikit-learn uses it by default (`criterion='gini'`), and we set `criterion='entropy'` today to match the slides
* on our 1,000 supernovae the first cut is 0.0147 (Gini) against 0.0159 (entropy)

## EXTRA: AdaBoost gives the misclassified objects more weight in the next tree

* start with equal weights on every training object, fit a shallow tree $h_t$, and find its weighted error $e_t$
* give that tree a vote $\alpha_t = \frac{1}{2} \ln\frac{1 - e_t}{e_t}$
* raise the weight of every object it got wrong, $w_i \to w_i\, e^{-\alpha_t y_i h_t(x_i)} / Z_t$ (with $y_i, h_t = \pm 1$, and $Z_t$ making the weights sum to 1), and fit the next tree
* final answer: the sign of $\sum_t \alpha_t h_t(x)$
* gradient boosting (our plot) fits each new tree to the negative gradient of the loss (for squared error, the residuals) instead